In [1]:
import numpy as np
from tabulate import tabulate
from magres.atoms import MagresAtoms
atoms = MagresAtoms.load_magres('/home/shiva/WMU/PhD/Scripts/phd_project/Python/NMR/CASTEP/8-HQ-ipc2-B_opt_magres_updated.magres')

In [2]:
def Rabc(alfa1, beta1, gama1): #Euler Rotation Matrix
    U = np.zeros((3, 3))
    #Changing input angles from degrees to radians
    alfa1 = np.radians(alfa1) 
    beta1 = np.radians(beta1)
    gama1 = np.radians(gama1)
    
    U[0, 0] = np.cos(alfa1)*np.cos(beta1)*np.cos(gama1) - np.sin(alfa1)*np.sin(gama1)
    U[0, 1] = np.sin(alfa1)*np.cos(beta1)*np.cos(gama1) + np.cos(alfa1)*np.sin(gama1)
    U[0, 2] = -np.sin(beta1)*np.cos(gama1)
    U[1, 0] = -np.cos(alfa1)*np.cos(beta1)*np.sin(gama1) - np.sin(alfa1)*np.cos(gama1)
    U[1, 1] = -np.sin(alfa1)*np.cos(beta1)*np.sin(gama1) + np.cos(alfa1)*np.cos(gama1)
    U[1, 2] = np.sin(beta1)*np.sin(gama1)
    U[2, 0] = np.cos(alfa1)*np.sin(beta1)
    U[2, 1] = np.sin(alfa1)*np.sin(beta1)
    U[2, 2] = np.cos(beta1)
    return U

In [3]:
# Sort eigenvalues of tensors as per the convention defined in article
def sort_eigenvalues(Tensor):
    #Calculate Quadrupolar Tensor in PAS
    eigenvalues, eigenvectors = np.linalg.eig(Tensor)
    print(' Unsorted Eigenvalues:\n', eigenvalues, '\n')
    print(' Unsorted Eigenvectors:\n', eigenvectors, '\n')

    avg_tensor = np.mean(eigenvalues) # Tr(A)Quad/3
    eigenvalue_diff = eigenvalues - avg_tensor

    # Get the indices of the sorted eigenvalues based on the absolute values
    sorted_indices = np.argsort(np.abs(eigenvalue_diff))

    # Sort both eigenvalues and eigenvectors using the sorted indices
    sorted_eigenvalues = eigenvalues[sorted_indices]
    sorted_eigenvectors = eigenvectors[:, sorted_indices] # The normalized (unit “length”) eigenvectors, 
    #                                                       such that the column eigenvectors[:,i] is the eigenvector corresponding to the eigenvalue eigenvalues[i]
    # eigenvectors need to be arranged so that first column for direction cosine matrix is eigenvector for x, second column is for y and third column is for z
    y_dc = sorted_eigenvectors[:,0]
    x_dc = sorted_eigenvectors[:,1]
    z_dc = sorted_eigenvectors[:,2]

    dc = np.stack((x_dc, y_dc, z_dc), axis = 1)


    print('Sorted Eigenvalues: \n', sorted_eigenvalues, '\n')
    print('Sorted Eigenvectors: \n', sorted_eigenvectors, '\n')
    return sorted_eigenvalues, dc,  avg_tensor, eigenvalues, eigenvectors

In [4]:
def get_euler_angles(eigenvectors):
    b = np.degrees(np.arccos(eigenvectors[2,2]))
    a = np.degrees(np.arctan(eigenvectors[2,1]/eigenvectors[2,0]))
    g = np.degrees(np.arctan(-eigenvectors[1,2]/eigenvectors[0,2]))

    return a, b, g
    

In [5]:
for atom in atoms.species('B'):
    print (atom, "sigma:\n",atom.ms.sigma)
    print()

11B1 sigma:
 [[90.04399411  2.4886535   1.24529086]
 [-2.10686887 83.75632904  6.96791588]
 [-4.14080973  2.15196889 78.09956279]]

11B2 sigma:
 [[90.04399411 -2.4886535   1.24529086]
 [ 2.10686887 83.75632904 -6.96791588]
 [-4.14080973 -2.15196889 78.09956279]]

11B3 sigma:
 [[90.04399411  2.4886535   1.24529086]
 [-2.10686887 83.75632904  6.96791588]
 [-4.14080973  2.15196889 78.09956279]]

11B4 sigma:
 [[90.04399411 -2.4886535   1.24529086]
 [ 2.10686887 83.75632904 -6.96791588]
 [-4.14080973 -2.15196889 78.09956279]]



In [6]:
for atom in atoms.species('B'):
    print (atom, "sigma:\n",atom.efg.Cq)
    print()

11B1 sigma:
 2.09926544904323

11B2 sigma:
 2.0992654490435108

11B3 sigma:
 2.0992654490432323

11B4 sigma:
 2.099265449043513



In [7]:
Cs = np.zeros((3, 3))                                # CS symmetric (l = 0 + l = 2) Tensor from magres
Cs[0,0] = 90.04399411; Cs[0,1] = -0.1909; Cs[0,2] = -1.44775;
Cs[1,0] = Cs[0,1]; Cs[1,1] = 83.75632904; Cs[1,2] = -4.55995;
Cs[2,0] = Cs[0,2]; Cs[2,1] = Cs[1,2]; Cs[2,2] = 78.09956279;

CS_total = np.zeros((3,3))                            # CS total shielding tensor ( l = 0 + l = 1 + l = 2) Tensor from magres
CS_total[0,0] = 90.04399411; CS_total[0,1] = -2.4887; CS_total[0,2] = 1.2453;
CS_total[1,0] = 2.1069; CS_total[1,1] = 83.75632904; CS_total[1,2] = -6.9679;
CS_total[2,0] = -4.1408; CS_total[2,1] = -2.1520; CS_total[2,2] = 78.0996;


efg = np.zeros((3, 3 ))                             # EFG Tensor from magres
efg[0,0]= -0.0285; efg[0,1]= -0.1852; efg[0,2]= 0.1112;
efg[1,0]= efg[0,1]; efg[1,1]= -0.0059; efg[1,2]= -0.0266;
efg[2,0]= efg[0,2]; efg[2,1]= efg[1,2]; efg[2,2]= 0.0344;

# Convert a.u. units to MHz
# Q tensor elements (MHz) = efg tensor (a.u.)* Q (barn) * 234.9647 
# Q = 0.04059 barn https://www-nds.iaea.org/publications/indc/indc-nds-0650.pdf

Q = 0.04059
V = efg*Q*234.9647

print(V)

[[-0.27181069 -1.76629262  1.06053855]
 [-1.76629262 -0.05626958 -0.25368998]
 [ 1.06053855 -0.25368998  0.32808027]]


In [14]:
sorted_eigenvalues_efg, dc_efg, quad_avg, eigenvalues_efg, eigenvectors_efg = sort_eigenvalues(V)
print('==================================\n')
sorted_eigenvalues_cs, dc_cs, cs_avg, eigenvalues_cs, eigenvectors_cs = sort_eigenvalues(Cs)

 Unsorted Eigenvalues:
 [ 2.09935196 -2.09646786 -0.0028841 ] 

 Unsorted Eigenvectors:
 [[-0.65236148  0.74551127 -0.13651906]
 [ 0.59045898  0.61284764  0.52514375]
 [-0.47516596 -0.26197464  0.83999202]] 

Sorted Eigenvalues: 
 [-0.0028841  -2.09646786  2.09935196] 

Sorted Eigenvectors: 
 [[-0.13651906  0.74551127 -0.65236148]
 [ 0.52514375  0.61284764  0.59045898]
 [ 0.83999202 -0.26197464 -0.47516596]] 


 Unsorted Eigenvalues:
 [75.43559489 90.24266624 86.22162481] 

 Unsorted Eigenvectors:
 [[-0.09272007 -0.9867383   0.13323106]
 [-0.48015165 -0.07291117 -0.87415008]
 [-0.87227139  0.14502237  0.46702369]] 

Sorted Eigenvalues: 
 [86.22162481 90.24266624 75.43559489] 

Sorted Eigenvectors: 
 [[ 0.13323106 -0.9867383  -0.09272007]
 [-0.87415008 -0.07291117 -0.48015165]
 [ 0.46702369  0.14502237 -0.87227139]] 



In [9]:

#Calculate Quadrupolar Tensor in PAS

Vyy = sorted_eigenvalues_efg[0]
Vxx = sorted_eigenvalues_efg[1]
Vzz = sorted_eigenvalues_efg[2]

print('Quadupolar Tensor Components Vyy, Vxx, Vzz: \n', Vyy, Vxx, Vzz)

print('================================================================================================')

#Calculate CSA Tensor in PAS

Csyy = sorted_eigenvalues_cs[0] 
Csxx = sorted_eigenvalues_cs[1]  
Cszz = sorted_eigenvalues_cs[2]



print('CSA Tensor Components δyy, δxx, δzz: \n', Csyy, Csxx, Cszz)

Quadupolar Tensor Components Vyy, Vxx, Vzz: 
 -0.002884100053809935 -2.0964678604447307 2.0993519604985402
CSA Tensor Components δyy, δxx, δzz: 
 86.22162481056866 90.24266623583684 75.43559489359455


In [10]:
iso_cs = (Csxx + Csyy + Cszz)/3

csa = Cszz - iso_cs
etas = (Csyy - Csxx)/csa

#for Quadrupolar
CQ_fit = Vzz

etaq = (Vyy - Vxx)/Vzz

table = [['CQ (MHz)', CQ_fit], ['etaq', etaq ], ['iso_cs (ppm)',iso_cs ],['csa (ppm)', csa],  ['etas', etas]  ]
print(tabulate(table, headers=['Qauntity', 'Fit Value']))

Qauntity        Fit Value
------------  -----------
CQ (MHz)         2.09935
etaq             0.997252
iso_cs (ppm)    83.9666
csa (ppm)       -8.53103
etas             0.471343


In [16]:
# Calculation for efg tensor
print('Direction cosine efg:\n')
print(dc_efg, '\n')
a_efg, b_efg, g_efg = get_euler_angles(dc_efg)

print("Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:")
print(a_efg, b_efg, g_efg, '\n')

print('=========================')
print('Direction cosine csa: \n')
print(dc_cs, '\n')
a_cs, b_cs, g_cs = get_euler_angles(dc_cs)

print("Calculated Euler angles (degrees) CSA PAS --> Crystal:")
print(a_cs, b_cs, g_cs, '\n')

Direction cosine efg:

[[ 0.74551127 -0.13651906 -0.65236148]
 [ 0.61284764  0.52514375  0.59045898]
 [-0.26197464  0.83999202 -0.47516596]] 

Calculated Euler angles (degrees) Quadrupolar PAS --> Crystal:
-72.67847249089047 118.37015646840118 42.14856402119201 

Direction cosine csa: 

[[-0.9867383   0.13323106 -0.09272007]
 [-0.07291117 -0.87415008 -0.48015165]
 [ 0.14502237  0.46702369 -0.87227139]] 

Calculated Euler angles (degrees) CSA PAS --> Crystal:
72.74914284275691 150.72367258805087 -79.07038137014605 



In [18]:
# Euler Matrix to relate Quadrupolar and CSA tensor

CSA_Q = np.matmul(np.linalg.inv(dc_efg), (dc_cs))

psi, chi, xi = get_euler_angles(CSA_Q)

print("Calculated Euler angles (degrees) PAS CSA --> Quadrupole:")
print('psi:', psi, 'chi:', chi, 'xi:', xi, '\n')

Calculated Euler angles (degrees) PAS CSA --> Quadrupole:
psi: -57.19572868476751 chi: 78.96253497789097 xi: -82.10185334181249 



**Rotation of tensors Crystal--> Tenon Frame**

In [13]:
#Euler angles Crystal--> Tenon Frame for LHQ
# Angles from Xray analysis
alpha = 280
beta = 45
gamma = 180
U  = Rabc(alpha, beta, gamma)
Cs_tenon = np.matmul(np.matmul(np.linalg.inv(U), Cs), U)

V_tenon = np.matmul(np.matmul(np.linalg.inv(U), V), U)

print('CSA Tensor in Crystal Frame: \n', Cs) #from magres
print('CSA Tensor in Tenon Frame: \n', Cs_tenon)

print('==========================')
print('Quad Tensor in Crystal Frame: \n', V)  #from magres
print('Quad Tensor in Tenon Frame: \n', V_tenon)

CSA Tensor in Crystal Frame: 
 [[90.04399411 -0.1909     -1.44775   ]
 [-0.1909     83.75632904 -4.55995   ]
 [-1.44775    -4.55995    78.09956279]]
CSA Tensor in Tenon Frame: 
 [[84.86612787 -3.20459703  2.27125769]
 [-3.20459703 84.40972962  6.46483072]
 [ 2.27125769  6.46483072 82.62402845]]
Quad Tensor in Crystal Frame: 
 [[-0.27181069 -1.76629262  1.06053855]
 [-1.76629262 -0.05626958 -0.25368998]
 [ 1.06053855 -0.25368998  0.32808027]]
Quad Tensor in Tenon Frame: 
 [[-0.45151866  1.1719973   1.45872863]
 [ 1.1719973  -0.63715468 -0.04735941]
 [ 1.45872863 -0.04735941  1.08867334]]
